# 00 — yield + send: Generators That RECEIVE Data

**Goal:** Understand `.send()` — the mechanism that turns generators into coroutines.

## The shift

So far, generators only OUTPUT data (with `yield`).
But `yield` is actually a TWO-WAY street:

```python
value = yield output    # yield sends output OUT, and receives input IN
```

- `yield output` → gives `output` to the caller
- `value = yield` → receives `value` from the caller (via `.send()`)

This turns a generator into a **coroutine** — a function that can both produce AND consume data.

## Exercise 0.1 — Basic send

Write an `echo()` generator:
- `while True:` loop
- `received = yield` — pause and wait for `.send()`
- Print what it received

Then:
1. Create the generator: `gen = echo()`
2. Prime it: `next(gen)` — advance to the first yield (**required** before `.send()`)
3. Send values: `gen.send("hello")`, `gen.send("world")`, `gen.send(42)`

**Why do you need `next(gen)` first?** The generator hasn't started yet — it's sitting at the beginning of the function, not at a `yield`. You need to advance it to the first `yield` point so it's ready to receive.

In [ ]:
# Exercise 0.1 — your code here


## Exercise 0.2 — Two-way communication

`yield` can both SEND and RECEIVE at the same time.

Write an `accumulator()` generator:
- Keeps a running `total` (starts at 0)
- `value = yield total` — yields current total OUT, receives next value IN
- `total += value`

Test:
```
next(acc)       → 0  (initial total)
acc.send(10)    → 10
acc.send(20)    → 30
acc.send(5)     → 35
```

Trace `acc.send(10)` step by step:
1. Where is the generator paused?
2. What does `.send(10)` do to the paused `yield`?
3. What code runs before it pauses again?
4. What does `.send(10)` return to the caller?

In [ ]:
# Exercise 0.2 — your code here


*Your trace of `.send(10)`:*


## Exercise 0.3 — Running average

Write a `running_average()` coroutine that:
- Receives numbers via `.send()`
- Yields the running average after each number

Test:
```
Send 10 → 10.0
Send 20 → 15.0
Send 30 → 20.0
Send 40 → 25.0
Send 50 → 30.0
```

In [ ]:
# Exercise 0.3 — your code here


## Exercise 0.4 — State machine as a coroutine

Write a `traffic_light()` coroutine that cycles RED → GREEN → YELLOW → RED → ...

The state is encoded in WHERE the execution is paused — NOT in a state variable. Each `yield` returns the current color. Send `"next"` to advance.

```python
light = traffic_light()
next(light)              # 'RED'
light.send('next')       # 'GREEN'
light.send('next')       # 'YELLOW'
light.send('next')       # 'RED'   (loops!)
```

**Key insight:** There are NO state variables. The state IS the position in the code — which `yield` the generator is paused at. Same way `async def` stores state.

In [ ]:
# Exercise 0.4 — your code here


## Exercise 0.5 — Coroutine-based grep (push model)

Write `coroutine_grep(pattern)` — receives lines via `.send()`, prints only matches.

This is the **push-based** version of the pipeline from Module 00.

```python
g = coroutine_grep('ERROR')
next(g)  # prime
g.send('ERROR: disk full')   # prints
g.send('INFO: all good')     # silent
g.send('ERROR: timeout')     # prints
```

In [ ]:
# Exercise 0.5 — your code here


## Exercise 0.6 — Closing and throwing

Write a `careful_worker()` coroutine that:
- Loops receiving values via `yield`
- Catches `GeneratorExit` (from `.close()`) and prints a cleanup message
- Catches `ValueError` (from `.throw()`) and prints the error

Test both `.close()` and `.throw(ValueError, "bad input!")` on separate instances.

**Connection to async:**
```
Generator             Async equivalent
─────────             ────────────────
next(gen)          →  event loop resumes coroutine
gen.send(val)      →  event loop sends result to coroutine  
gen.throw(exc)     →  task.cancel() / exception propagation
gen.close()        →  task cleanup
yield              →  await
StopIteration      →  coroutine returns
```

In [ ]:
# Exercise 0.6 — your code here


## Summary

| Method | Direction | Async equivalent |
|--------|-----------|------------------|
| `yield value` | Generator → Caller | Coroutine suspends, event loop gets control |
| `value = yield` | Caller → Generator | Event loop resumes coroutine with result |
| `.send(data)` | Push data into generator | Event loop delivers I/O result |
| `.throw(exc)` | Push exception into generator | `task.cancel()` |
| `.close()` | Shut down generator | Graceful coroutine cleanup |

**Next notebook:** `yield from` — delegating to sub-generators (the precursor to `await`)